# 02 — Visit Risk Classification (Model A)

**Target variable:** `risk_score` (Low=0, Medium=1, High=2)  
**Dataset:** `data/processed/model_table_risk.csv` (24,210 rows × 28 columns)  
**Split:** 70% train / 15% validation / 15% test (test held out — evaluated in Phase 4)  
**Models:** Logistic Regression (baseline) · XGBoost (base) · XGBoost (tuned)  
**Class imbalance:** Low 49.9% / Medium 29.9% / High 20.2% → handled via `class_weight='balanced'` and XGBoost sample weights

**Phase 3 scope:** Train models, report train + validation metrics, save artifacts.  
Test set evaluation and explainability are covered in Phase 4.

> **Note on Model A:** EDA confirmed that `risk_score` has no real predictors in this synthetic dataset.  
> Near-baseline performance (~49.9% accuracy) is expected. XGBoost tuning is included for completeness  
> but will not meaningfully improve validation performance — it reduces overfitting without raising the ceiling.  
> This is a data constraint, not a modeling failure. Document in the Phase 4 model card.

## 1. Imports & Configuration

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
os.makedirs('models', exist_ok=True)

print('All imports successful.')
print('Models will be saved to: models/')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('data/processed/model_table_risk.csv')

TARGET_MAP = {0: 'Low', 1: 'Medium', 2: 'High'}

print(f'Shape : {df.shape}')
print('\nTarget distribution:')
for k, v in df['risk_score'].value_counts().sort_index().items():
    print(f'  {TARGET_MAP[k]:8s} ({k}): {v:,}  ({v/len(df)*100:.1f}%)')

## 3. Train / Validation / Test Split (70 / 15 / 15)

Stratified splitting ensures class proportions are preserved across all three sets.  
The test set is isolated immediately and **not used anywhere in Phase 3**.

In [ ]:
FEATURE_COLS = [c for c in df.columns if c != 'risk_score']
TARGET_COL   = 'risk_score'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

# Step 1: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

# Step 2: split temp equally → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f'Train : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Val   : {X_val.shape[0]:,} rows ({X_val.shape[0]/len(df)*100:.1f}%)')
print(f'Test  : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(df)*100:.1f}%) <- held out for Phase 4')

print('\nClass distribution check (stratification):')
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = y_split.value_counts(normalize=True).sort_index()
    dist   = '  '.join([f'{TARGET_MAP[k]}: {v:.1%}' for k, v in counts.items()])
    print(f'  {name:6s} -> {dist}')

## 4. Feature Scaling

StandardScaler is fit **only on training data** and then applied to val and test sets.  
Fitting on val/test would constitute data leakage.  
Numeric columns scaled: `age`, `length_of_stay_hours`, `billed_amount`.  
Note: `payment_days` is absent from Model A — excluded due to temporal leakage (billing hasn't occurred at visit-time risk scoring).

In [ ]:
NUMERIC_COLS = ['age', 'length_of_stay_hours', 'billed_amount']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[NUMERIC_COLS] = scaler.fit_transform(X_train[NUMERIC_COLS])
X_val_scaled[NUMERIC_COLS]   = scaler.transform(X_val[NUMERIC_COLS])
X_test_scaled[NUMERIC_COLS]  = scaler.transform(X_test[NUMERIC_COLS])

joblib.dump(scaler, 'models/risk_scaler.joblib')

print(f'Scaled columns : {NUMERIC_COLS}')
print(f'Scaler saved   -> models/risk_scaler.joblib')

## 5. Helper — Metrics Reporter

Reusable function for printing metrics and plotting confusion matrices.  
Used consistently across all three models in this notebook.

In [ ]:
CLASS_NAMES = ['Low', 'Medium', 'High']

def report_metrics(model_name, split_name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f'\n-- {model_name} | {split_name} --')
    print(f'  Accuracy           : {acc:.4f}')
    print(f'  Weighted Precision : {prec:.4f}')
    print(f'  Weighted Recall    : {rec:.4f}')
    print(f'  Weighted F1        : {f1:.4f}')
    print(f'\n  Per-class report:')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 4))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'{model_name} — {split_name} Confusion Matrix')
    plt.tight_layout()
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

print('report_metrics() defined.')

## 6. Model A1 — Logistic Regression (Baseline)

`class_weight='balanced'` adjusts loss contributions inversely proportional to class frequency,  
preventing the majority class (Low, 49.9%) from dominating training.  
`max_iter=1000` ensures convergence on this dataset size.

In [ ]:
lr_risk = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

lr_risk.fit(X_train_scaled, y_train)
print('Logistic Regression training complete.')

### 6.1 Cross-Validation on Training Set (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores_lr = cross_val_score(
    lr_risk, X_train_scaled, y_train,
    cv=cv, scoring='f1_weighted', n_jobs=-1
)

print('Logistic Regression — 5-Fold CV (weighted F1):')
print(f'  Fold scores : {[round(s, 4) for s in cv_scores_lr]}')
print(f'  Mean        : {cv_scores_lr.mean():.4f}')
print(f'  Std         : {cv_scores_lr.std():.4f}')

### 6.2 Train & Validation Metrics

In [ ]:
lr_train_preds = lr_risk.predict(X_train_scaled)
lr_val_preds   = lr_risk.predict(X_val_scaled)

lr_train_metrics = report_metrics('Logistic Regression', 'Train', y_train, lr_train_preds)
lr_val_metrics   = report_metrics('Logistic Regression', 'Validation', y_val, lr_val_preds)

### 6.3 Save Artifact

In [ ]:
joblib.dump(lr_risk, 'models/risk_logreg.joblib')
print('Saved -> models/risk_logreg.joblib')

## 7. Model A2 — XGBoost (Base)

XGBoost does not natively support `class_weight='balanced'` for multiclass.  
`compute_sample_weight` produces per-sample weights that upweight minority class  
samples during boosting — equivalent effect.

In [ ]:
sample_weights_risk = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_risk = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_risk.fit(
    X_train_scaled, y_train,
    sample_weight=sample_weights_risk,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

print('XGBoost (base) training complete.')

### 7.1 Cross-Validation on Training Set (5-Fold Stratified)

In [ ]:
# Manual CV loop required for XGBoost with sample weights
# (sklearn 1.8 removed fit_params from cross_val_score)
xgb_cv_f1_scores = []

for train_idx, val_idx in cv.split(X_train_scaled, y_train):
    Xf = X_train_scaled.iloc[train_idx]
    Xv = X_train_scaled.iloc[val_idx]
    yf = y_train.iloc[train_idx]
    yv = y_train.iloc[val_idx]
    sw_fold = compute_sample_weight('balanced', yf)

    m = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    m.fit(Xf, yf, sample_weight=sw_fold, verbose=False)
    xgb_cv_f1_scores.append(f1_score(yv, m.predict(Xv), average='weighted'))

cv_scores_xgb = np.array(xgb_cv_f1_scores)

print('XGBoost (base) — 5-Fold CV (weighted F1):')
print(f'  Fold scores : {[round(s, 4) for s in cv_scores_xgb]}')
print(f'  Mean        : {cv_scores_xgb.mean():.4f}')
print(f'  Std         : {cv_scores_xgb.std():.4f}')

### 7.2 Train & Validation Metrics

In [ ]:
xgb_train_preds = xgb_risk.predict(X_train_scaled)
xgb_val_preds   = xgb_risk.predict(X_val_scaled)

xgb_train_metrics = report_metrics('XGBoost (base)', 'Train', y_train, xgb_train_preds)
xgb_val_metrics   = report_metrics('XGBoost (base)', 'Validation', y_val, xgb_val_preds)

### 7.3 Save Base XGBoost Artifact

In [ ]:
joblib.dump(xgb_risk, 'models/risk_xgb.joblib')
print('Saved -> models/risk_xgb.joblib')

## 8. Model A3 — XGBoost (Tuned via RandomizedSearchCV)

> **Important caveat for Model A:**  
> Tuning reduces the train/val gap (overfitting) but does **not** raise the validation F1 ceiling.  
> The ceiling is set by the data — `risk_score` has no real predictors in this synthetic dataset.  
> The tuned model is a more honest model, not a better one. Document this in the Phase 4 model card.

**Parameter search space:**
- `n_estimators`: number of boosting rounds
- `max_depth`: tree depth (lower = less overfit)
- `learning_rate`: step size shrinkage
- `subsample`: fraction of training rows per tree
- `colsample_bytree`: fraction of features per tree
- `min_child_weight`: minimum sum of instance weight in a leaf (higher = more conservative)
- `gamma`: minimum loss reduction to make a split (higher = more conservative)
- `reg_alpha`: L1 regularisation
- `reg_lambda`: L2 regularisation

In [ ]:
param_dist_risk = {
    'n_estimators'    : randint(100, 500),
    'max_depth'       : randint(3, 8),
    'learning_rate'   : uniform(0.01, 0.2),
    'subsample'       : uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),
    'gamma'           : uniform(0, 0.5),
    'reg_alpha'       : uniform(0, 1.0),
    'reg_lambda'      : uniform(0.5, 2.0),
}

xgb_base_for_search = XGBClassifier(
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rs_risk = RandomizedSearchCV(
    estimator=xgb_base_for_search,
    param_distributions=param_dist_risk,
    n_iter=50,
    scoring='f1_weighted',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rs_risk.fit(
    X_train_scaled, y_train,
    sample_weight=sample_weights_risk
)

print(f'\nBest CV F1 (weighted) : {rs_risk.best_score_:.4f}')
print(f'Best parameters:')
for k, v in rs_risk.best_params_.items():
    print(f'  {k:20s}: {v}')

### 8.1 Train Tuned Model with Best Parameters

In [ ]:
xgb_risk_tuned = XGBClassifier(
    **rs_risk.best_params_,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_risk_tuned.fit(
    X_train_scaled, y_train,
    sample_weight=sample_weights_risk,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

print('XGBoost (tuned) training complete.')

### 8.2 Train & Validation Metrics

In [ ]:
xgb_tuned_train_preds = xgb_risk_tuned.predict(X_train_scaled)
xgb_tuned_val_preds   = xgb_risk_tuned.predict(X_val_scaled)

xgb_tuned_train_metrics = report_metrics('XGBoost (tuned)', 'Train', y_train, xgb_tuned_train_preds)
xgb_tuned_val_metrics   = report_metrics('XGBoost (tuned)', 'Validation', y_val, xgb_tuned_val_preds)

### 8.3 Save Tuned XGBoost Artifact

In [ ]:
joblib.dump(xgb_risk_tuned, 'models/risk_xgb_tuned.joblib')
print('Saved -> models/risk_xgb_tuned.joblib')

## 9. Model Comparison Summary

In [ ]:
summary = pd.DataFrame({
    'Model' : [
        'Logistic Regression', 'Logistic Regression',
        'XGBoost (base)',      'XGBoost (base)',
        'XGBoost (tuned)',     'XGBoost (tuned)'
    ],
    'Split' : ['Train', 'Val', 'Train', 'Val', 'Train', 'Val'],
    'Accuracy': [
        lr_train_metrics['accuracy'],        lr_val_metrics['accuracy'],
        xgb_train_metrics['accuracy'],       xgb_val_metrics['accuracy'],
        xgb_tuned_train_metrics['accuracy'], xgb_tuned_val_metrics['accuracy']
    ],
    'Precision': [
        lr_train_metrics['precision'],        lr_val_metrics['precision'],
        xgb_train_metrics['precision'],       xgb_val_metrics['precision'],
        xgb_tuned_train_metrics['precision'], xgb_tuned_val_metrics['precision']
    ],
    'Recall': [
        lr_train_metrics['recall'],        lr_val_metrics['recall'],
        xgb_train_metrics['recall'],       xgb_val_metrics['recall'],
        xgb_tuned_train_metrics['recall'], xgb_tuned_val_metrics['recall']
    ],
    'F1 (Weighted)': [
        lr_train_metrics['f1'],        lr_val_metrics['f1'],
        xgb_train_metrics['f1'],       xgb_val_metrics['f1'],
        xgb_tuned_train_metrics['f1'], xgb_tuned_val_metrics['f1']
    ]
}).round(4)

print('-- Model A: Risk Score Classifier — Performance Summary --')
print(summary.to_string(index=False))

print('\n-- Cross-Validation Summary (5-Fold, weighted F1 on train) --')
print(f'  Logistic Regression : {cv_scores_lr.mean():.4f} +/- {cv_scores_lr.std():.4f}')
print(f'  XGBoost (base)      : {cv_scores_xgb.mean():.4f} +/- {cv_scores_xgb.std():.4f}')
print(f'  XGBoost (tuned)     : {rs_risk.best_score_:.4f} (RandomizedSearchCV best CV score)')

print('\n-- Overfitting Analysis --')
base_gap  = xgb_train_metrics["f1"] - xgb_val_metrics["f1"]
tuned_gap = xgb_tuned_train_metrics["f1"] - xgb_tuned_val_metrics["f1"]
print(f'  XGBoost base  train/val F1 gap : {base_gap:.4f}')
print(f'  XGBoost tuned train/val F1 gap : {tuned_gap:.4f}')
print(f'  Gap reduction                  : {base_gap - tuned_gap:.4f}')

print('\n-- Model Card Note --')
print('  Tuning reduced the train/val gap (overfitting) but did NOT raise the')
print('  validation F1 ceiling. risk_score has no real predictors in this')
print('  synthetic dataset. This is a data constraint, not a modeling failure.')
print(f'  Majority class baseline accuracy: ~49.9%')

## 10. Feature Schema

In [ ]:
feature_schema_risk = {
    'model'            : 'Model A — Visit Risk Classifier',
    'target'           : 'risk_score',
    'target_encoding'  : {'Low': 0, 'Medium': 1, 'High': 2},
    'n_features'       : len(FEATURE_COLS),
    'features'         : FEATURE_COLS,
    'numeric_features' : NUMERIC_COLS,
    'scaled_with'      : "StandardScaler (fit on train set only)",
    'scaler_path'      : 'models/risk_scaler.joblib',
    'encoding'         : "One-Hot Encoding with drop='first'",
    'excluded_features': {
        'leakage'          : ['approved_amount', 'approval_ratio'],
        'temporal_leakage' : ['payment_days', 'payment_days_band'],
        'other_target'     : ['claim_status']
    },
    'class_imbalance_handling': "class_weight='balanced' (LogReg) / compute_sample_weight (XGBoost)",
    'split': {'train': int(X_train.shape[0]), 'val': int(X_val.shape[0]), 'test': int(X_test.shape[0])},
    'artifacts': {
        'logistic_regression': 'models/risk_logreg.joblib',
        'xgboost_base'       : 'models/risk_xgb.joblib',
        'xgboost_tuned'      : 'models/risk_xgb_tuned.joblib',
        'scaler'             : 'models/risk_scaler.joblib'
    },
    'tuning': {
        'method'    : 'RandomizedSearchCV',
        'n_iter'    : 50,
        'cv_folds'  : 5,
        'scoring'   : 'f1_weighted',
        'best_params': rs_risk.best_params_,
        'best_cv_f1' : round(rs_risk.best_score_, 4)
    },
    'caveats': [
        'Dataset is synthetic — risk_score has no real predictors (EDA confirmed).',
        'Near-baseline performance (~49.9%) is expected. Document in model card.',
        'Tuning reduced overfitting but did not raise the validation F1 ceiling.',
        'payment_days excluded due to temporal leakage — not available at visit time.'
    ]
}

with open('models/feature_schema_risk.json', 'w') as f:
    json.dump(feature_schema_risk, f, indent=2)

print('Saved -> models/feature_schema_risk.json')
print(json.dumps(feature_schema_risk, indent=2))

## 11. Save Test Set (Held Out for Phase 4)

In [ ]:
test_df = pd.DataFrame(X_test_scaled.values, columns=FEATURE_COLS)
test_df['risk_score'] = y_test.values
test_df.to_csv('data/processed/test_risk.csv', index=False)

print(f'Test set saved -> data/processed/test_risk.csv')
print(f'Shape          : {test_df.shape}')
print('Held out — used only in Phase 4 evaluation.')